In [ ]:
"""
Speech Intelligibility Assessment Using Whisper Large V2
=========================================================

Implementation inspired by:
Zhang, V.W., Sebastian, A., & Monaghan, J.J.M. (2025). 
"Automated Speech Intelligibility Assessment Using AI-Based Transcription 
in Children with Cochlear Implants, Hearing Aids, and Normal Hearing."
J. Clin. Med. 2025, 14, 5280.
https://doi.org/10.3390/jcm14155280

This tool evaluates speech intelligibility by:
1. Transcribing speech using Whisper Large V2 from Hugging Face
2. Comparing AI transcriptions with naive listener transcriptions
3. Calculating Word Error Rate (WER)
4. Performing Bland-Altman analysis for agreement assessment
"""

# ============================================================================
# IMPORTS AND DEPENDENCIES
# ============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# For audio processing and Whisper model
try:
    import torch
    import torchaudio
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
except ImportError:
    print("Please install required packages:")
    print("pip install torch torchaudio transformers")
    raise


# ============================================================================
# MAIN CLASS: SPEECH INTELLIGIBILITY ASSESSOR
# ============================================================================

class SpeechIntelligibilityAssessor:
    """
    Automated speech intelligibility assessment tool using Whisper Large V2.
    """
    
    # ------------------------------------------------------------------------
    # INITIALIZATION
    # ------------------------------------------------------------------------
    
    def __init__(self, model_name: str = "openai/whisper-large-v2"):
        """
        Initialize the speech intelligibility assessor.
        
        Args:
            model_name: Hugging Face model identifier for Whisper
        """
        print(f"Loading Whisper model: {model_name}")
        print("This may take a few moments on first run...")
        
        # Load Whisper model from Hugging Face
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        print("Model loaded successfully!\n")
    
    # ------------------------------------------------------------------------
    # AUDIO TRANSCRIPTION
    # ------------------------------------------------------------------------
    
    def transcribe_audio(self, audio_path: str, sample_rate: int = 16000) -> str:
        """
        Transcribe audio file using Whisper Large V2.
        
        Args:
            audio_path: Path to audio file
            sample_rate: Target sample rate (Whisper uses 16kHz)
            
        Returns:
            Transcribed text
        """
        # Load and resample audio
        waveform, sr = torchaudio.load(audio_path)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample if necessary
        if sr != sample_rate:
            resampler = torchaudio.transforms.Resample(sr, sample_rate)
            waveform = resampler(waveform)
        
        # Process audio
        input_features = self.processor(
            waveform.squeeze().numpy(),
            sampling_rate=sample_rate,
            return_tensors="pt"
        ).input_features
        
        # Generate transcription
        with torch.no_grad():
            input_features = input_features.to(self.device)
            predicted_ids = self.model.generate(input_features)
            transcription = self.processor.batch_decode(
                predicted_ids, 
                skip_special_tokens=True
            )[0]
        
        return transcription.strip()
    
    # ------------------------------------------------------------------------
    # WORD ERROR RATE (WER) CALCULATION
    # ------------------------------------------------------------------------
    
    def calculate_wer(self, reference: str, hypothesis: str) -> Tuple[float, Dict]:
        """
        Calculate Word Error Rate (WER) between reference and hypothesis.
        
        WER = (S + D + I) / N × 100
        where S=substitutions, D=deletions, I=insertions, N=total words
        
        Args:
            reference: Reference transcription (ground truth)
            hypothesis: Hypothesis transcription (predicted)
            
        Returns:
            Tuple of (WER percentage, error details dict)
        """
        ref_words = reference.lower().split()
        hyp_words = hypothesis.lower().split()
        
        # Dynamic programming for edit distance
        d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1), dtype=np.int32)
        
        for i in range(len(ref_words) + 1):
            d[i][0] = i
        for j in range(len(hyp_words) + 1):
            d[0][j] = j
        
        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                if ref_words[i-1] == hyp_words[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    substitution = d[i-1][j-1] + 1
                    insertion = d[i][j-1] + 1
                    deletion = d[i-1][j] + 1
                    d[i][j] = min(substitution, insertion, deletion)
        
        # Backtrack to count error types
        i, j = len(ref_words), len(hyp_words)
        substitutions = deletions = insertions = 0
        
        while i > 0 or j > 0:
            if i > 0 and j > 0 and ref_words[i-1] == hyp_words[j-1]:
                i -= 1
                j -= 1
            elif i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + 1:
                substitutions += 1
                i -= 1
                j -= 1
            elif j > 0 and d[i][j] == d[i][j-1] + 1:
                insertions += 1
                j -= 1
            else:
                deletions += 1
                i -= 1
        
        total_words = len(ref_words)
        wer = ((substitutions + deletions + insertions) / total_words * 100) if total_words > 0 else 0
        
        return wer, {
            'substitutions': substitutions,
            'deletions': deletions,
            'insertions': insertions,
            'total_words': total_words
        }
    
    # ------------------------------------------------------------------------
    # CORRECT WORDS COUNTING
    # ------------------------------------------------------------------------
    
    def count_correct_words(self, reference: str, hypothesis: str) -> int:
        """
        Count number of correctly transcribed words.
        
        Args:
            reference: Reference transcription
            hypothesis: Hypothesis transcription
            
        Returns:
            Number of correct words
        """
        ref_words = set(reference.lower().split())
        hyp_words = set(hypothesis.lower().split())
        return len(ref_words.intersection(hyp_words))
    
    # ------------------------------------------------------------------------
    # BLAND-ALTMAN ANALYSIS
    # ------------------------------------------------------------------------
    
    def bland_altman_analysis(self, ai_scores: np.ndarray, 
                             listener_scores: np.ndarray,
                             listener_name: str = "Naive Listener") -> Dict:
        """
        Perform Bland-Altman analysis to assess agreement between AI and listener.
        
        Args:
            ai_scores: Array of AI transcription scores (correct words)
            listener_scores: Array of listener transcription scores
            listener_name: Name of the listener for labeling
            
        Returns:
            Dictionary containing analysis results
        """
        # Calculate mean and difference
        means = (ai_scores + listener_scores) / 2
        diffs = ai_scores - listener_scores
        
        # Calculate statistics
        mean_diff = np.mean(diffs)
        std_diff = np.std(diffs, ddof=1)
        
        # 95% limits of agreement
        upper_loa = mean_diff + 1.96 * std_diff
        lower_loa = mean_diff - 1.96 * std_diff
        
        # Count points outside LOA
        outside_loa = np.sum((diffs > upper_loa) | (diffs < lower_loa))
        total_points = len(diffs)
        percent_outside = (outside_loa / total_points) * 100
        
        return {
            'mean_difference': mean_diff,
            'std_difference': std_diff,
            'upper_loa': upper_loa,
            'lower_loa': lower_loa,
            'outside_loa_count': outside_loa,
            'outside_loa_percent': percent_outside,
            'listener_name': listener_name
        }
    
    # ------------------------------------------------------------------------
    # BLAND-ALTMAN PLOTTING
    # ------------------------------------------------------------------------
    
    def plot_bland_altman(self, ai_scores: np.ndarray, 
                         listener_scores: np.ndarray,
                         listener_name: str = "Naive Listener",
                         save_path: str = None):
        """
        Create Bland-Altman plot.
        
        Args:
            ai_scores: Array of AI transcription scores
            listener_scores: Array of listener transcription scores
            listener_name: Name of the listener
            save_path: Path to save the plot (optional)
        """
        results = self.bland_altman_analysis(ai_scores, listener_scores, listener_name)
        
        means = (ai_scores + listener_scores) / 2
        diffs = ai_scores - listener_scores
        
        plt.figure(figsize=(10, 6))
        plt.scatter(means, diffs, alpha=0.6, s=50)
        
        # Mean difference line
        plt.axhline(results['mean_difference'], color='blue', 
                   linestyle='-', linewidth=2, 
                   label=f"Mean diff: {results['mean_difference']:.2f}")
        
        # Limits of agreement
        plt.axhline(results['upper_loa'], color='red', 
                   linestyle='--', linewidth=2,
                   label=f"Upper LOA: {results['upper_loa']:.2f}")
        plt.axhline(results['lower_loa'], color='red', 
                   linestyle='--', linewidth=2,
                   label=f"Lower LOA: {results['lower_loa']:.2f}")
        
        plt.xlabel('Average Number of Correct Words\n(AI Model + Naive Listener) / 2', 
                  fontsize=12)
        plt.ylabel('Difference in Correct Words\n(AI Model - Naive Listener)', 
                  fontsize=12)
        plt.title(f'Bland-Altman Plot: AI Model vs. {listener_name}\n' +
                 f'{results["outside_loa_percent"]:.1f}% points outside 95% LOA',
                 fontsize=14, fontweight='bold')
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Bland-Altman plot saved to: {save_path}")
        
        plt.show()


# ============================================================================
# BATCH PROCESSING FUNCTION
# ============================================================================

def process_audio_folder(assessor: SpeechIntelligibilityAssessor,
                        audio_folder: str,
                        reference_file: str,
                        listener_file: str,
                        output_folder: str = "results"):
    """
    Process folder of audio files and compare with naive listener transcriptions.
    
    Args:
        assessor: SpeechIntelligibilityAssessor instance
        audio_folder: Path to folder containing audio files
        reference_file: Excel file with reference transcriptions
        listener_file: Excel file with naive listener transcriptions
        output_folder: Folder to save results
    """
    # ------------------------------------------------------------------------
    # SETUP AND DATA LOADING
    # ------------------------------------------------------------------------
    
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Load reference and listener data
    print("Loading reference and listener transcriptions...")
    ref_df = pd.read_excel(reference_file)
    listener_df = pd.read_excel(listener_file)
    
    # Expected columns: 'audio_file', 'reference_text', 'listener_1', 'listener_2', 'listener_3'
    
    results = []
    
    # ------------------------------------------------------------------------
    # PROCESS EACH AUDIO FILE
    # ------------------------------------------------------------------------
    
    print("\nProcessing audio files...")
    for idx, row in ref_df.iterrows():
        audio_file = row['audio_file']
        reference = row['reference_text']
        
        audio_path = os.path.join(audio_folder, audio_file)
        
        if not os.path.exists(audio_path):
            print(f"Warning: Audio file not found: {audio_file}")
            continue
        
        print(f"Processing: {audio_file}")
        
        # AI transcription
        ai_transcription = assessor.transcribe_audio(audio_path)
        
        # Calculate WER and correct words for AI
        ai_wer, ai_errors = assessor.calculate_wer(reference, ai_transcription)
        ai_correct = assessor.count_correct_words(reference, ai_transcription)
        
        result = {
            'audio_file': audio_file,
            'reference': reference,
            'ai_transcription': ai_transcription,
            'ai_wer': ai_wer,
            'ai_correct_words': ai_correct,
            'ai_substitutions': ai_errors['substitutions'],
            'ai_deletions': ai_errors['deletions'],
            'ai_insertions': ai_errors['insertions']
        }
        
        # Process each naive listener
        listener_row = listener_df[listener_df['audio_file'] == audio_file].iloc[0]
        
        for i in range(1, 4):  # Assume 3 listeners
            listener_col = f'listener_{i}'
            if listener_col in listener_row:
                listener_trans = listener_row[listener_col]
                listener_wer, listener_errors = assessor.calculate_wer(
                    reference, listener_trans
                )
                listener_correct = assessor.count_correct_words(reference, listener_trans)
                
                result[f'listener_{i}_transcription'] = listener_trans
                result[f'listener_{i}_wer'] = listener_wer
                result[f'listener_{i}_correct_words'] = listener_correct
        
        results.append(result)
    
    # ------------------------------------------------------------------------
    # SAVE RESULTS AND GENERATE STATISTICS
    # ------------------------------------------------------------------------
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    # Save results
    output_file = os.path.join(output_folder, 'transcription_results.xlsx')
    results_df.to_excel(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
    # ------------------------------------------------------------------------
    # DISPLAY SUMMARY STATISTICS
    # ------------------------------------------------------------------------
    
    # Calculate and display summary statistics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    print(f"\nAI Model (Whisper Large V2):")
    print(f"  Mean WER: {results_df['ai_wer'].mean():.2f}%")
    print(f"  Std WER: {results_df['ai_wer'].std():.2f}%")
    print(f"  Mean Correct Words: {results_df['ai_correct_words'].mean():.2f}")
    
    for i in range(1, 4):
        if f'listener_{i}_wer' in results_df.columns:
            print(f"\nNaive Listener {i}:")
            print(f"  Mean WER: {results_df[f'listener_{i}_wer'].mean():.2f}%")
            print(f"  Std WER: {results_df[f'listener_{i}_wer'].std():.2f}%")
            print(f"  Mean Correct Words: {results_df[f'listener_{i}_correct_words'].mean():.2f}")
    
    # ------------------------------------------------------------------------
    # BLAND-ALTMAN ANALYSIS FOR EACH LISTENER
    # ------------------------------------------------------------------------
    
    # Bland-Altman analysis for each listener
    print("\n" + "="*60)
    print("BLAND-ALTMAN ANALYSIS")
    print("="*60)
    
    for i in range(1, 4):
        if f'listener_{i}_correct_words' in results_df.columns:
            ai_scores = results_df['ai_correct_words'].values
            listener_scores = results_df[f'listener_{i}_correct_words'].values
            
            ba_results = assessor.bland_altman_analysis(
                ai_scores, listener_scores, f"Naive Listener {i}"
            )
            
            print(f"\nAI Model vs. Naive Listener {i}:")
            print(f"  Mean Difference: {ba_results['mean_difference']:.2f}")
            print(f"  95% LOA: [{ba_results['lower_loa']:.2f}, {ba_results['upper_loa']:.2f}]")
            print(f"  Points outside LOA: {ba_results['outside_loa_count']} ({ba_results['outside_loa_percent']:.1f}%)")
            
            # Create Bland-Altman plot
            plot_path = os.path.join(output_folder, f'bland_altman_listener_{i}.png')
            assessor.plot_bland_altman(
                ai_scores, listener_scores, 
                f"Naive Listener {i}", 
                plot_path
            )
    
    return results_df


# ============================================================================
# MAIN EXECUTION (EXAMPLE USAGE)
# ============================================================================

# Example usage
if __name__ == "__main__":
    print("Speech Intelligibility Assessment Tool")
    print("=" * 60)
    print("Implementation based on Zhang et al. (2025)")
    print("Using Whisper Large V2 from Hugging Face")
    print("=" * 60 + "\n")
    
    # ------------------------------------------------------------------------
    # INITIALIZE ASSESSOR
    # ------------------------------------------------------------------------
    
    # Initialize assessor
    assessor = SpeechIntelligibilityAssessor()
    
    # ------------------------------------------------------------------------
    # EXAMPLE 1: BATCH PROCESSING
    # ------------------------------------------------------------------------
    
    # Example: Process audio files
    # Modify these paths according to your data structure
    """
    results_df = process_audio_folder(
        assessor=assessor,
        audio_folder="path/to/audio/files",
        reference_file="reference_transcriptions.xlsx",
        listener_file="naive_listener_transcriptions.xlsx",
        output_folder="results"
    )
    """
    
    # ------------------------------------------------------------------------
    # EXAMPLE 2: SINGLE FILE TRANSCRIPTION
    # ------------------------------------------------------------------------
    
    # Example: Single file transcription
    print("Example: Single file transcription")
    print("-" * 60)
    
    # Uncomment and modify for your audio file
    """
    audio_path = "example_audio.wav"
    reference = "The boy walked to the table"
    
    transcription = assessor.transcribe_audio(audio_path)
    print(f"Reference:      {reference}")
    print(f"AI Transcription: {transcription}")
    
    wer, errors = assessor.calculate_wer(reference, transcription)
    print(f"\nWER: {wer:.2f}%")
    print(f"Substitutions: {errors['substitutions']}")
    print(f"Deletions: {errors['deletions']}")
    print(f"Insertions: {errors['insertions']}")
    """
    
    print("\nSetup complete! Modify the paths in the script to process your data.")